# Spaceship Titanic Dataset with PyTorch and Decision Forests

This notebook implements a Random Forest solution using PyTorch ecosystem and sklearn.
Since PyTorch doesn't have native decision forest support, we'll use sklearn's RandomForestClassifier
with PyTorch for data handling and preprocessing.

The approach follows the same methodology as the TensorFlow Decision Forests solution.

# Import Libraries

In [ ]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.tree import plot_tree

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

# Load the Dataset

In [ ]:
# Load dataset into Pandas DataFrame
dataset_df = pd.read_csv('/kaggle/input/spaceship-titanic/train.csv')
print(f"Full train dataset shape is {dataset_df.shape}")

In [ ]:
# Display the first 5 examples
dataset_df.head(5)

# Basic Data Exploration

In [ ]:
dataset_df.describe()

In [ ]:
dataset_df.info()

# Bar Chart for Label Column: Transported

In [ ]:
plot_df = dataset_df.Transported.value_counts()
plot_df.plot(kind="bar")
plt.title("Distribution of Transported Label")
plt.ylabel("Count")
plt.show()

# Numerical Data Distribution

In [ ]:
fig, ax = plt.subplots(5, 1, figsize=(10, 10))
plt.subplots_adjust(top=2)

sns.histplot(dataset_df['Age'], color='b', bins=50, ax=ax[0])
sns.histplot(dataset_df['FoodCourt'], color='b', bins=50, ax=ax[1])
sns.histplot(dataset_df['ShoppingMall'], color='b', bins=50, ax=ax[2])
sns.histplot(dataset_df['Spa'], color='b', bins=50, ax=ax[3])
sns.histplot(dataset_df['VRDeck'], color='b', bins=50, ax=ax[4])
plt.show()

# Data Preprocessing

Following the same preprocessing steps as the TensorFlow solution

In [ ]:
# Drop PassengerId and Name columns
dataset_df = dataset_df.drop(['PassengerId', 'Name'], axis=1)
dataset_df.head(5)

In [ ]:
# Check for missing values
dataset_df.isnull().sum().sort_values(ascending=False)

In [ ]:
# Replace NaN values with 0 for VIP and CryoSleep
dataset_df[['VIP', 'CryoSleep']] = dataset_df[['VIP', 'CryoSleep']].fillna(value=0)

# Extract Deck, Cabin_num, and Side from Cabin column
dataset_df[["Deck", "Cabin_num", "Side"]] = dataset_df["Cabin"].str.split("/", expand=True)
dataset_df = dataset_df.drop('Cabin', axis=1)

# Convert boolean columns to integers
dataset_df['VIP'] = dataset_df['VIP'].astype(int)
dataset_df['CryoSleep'] = dataset_df['CryoSleep'].astype(int)
dataset_df['Transported'] = dataset_df['Transported'].astype(int)

dataset_df.head(5)

# Prepare Features and Labels

In [ ]:
# Separate features and labels
label = dataset_df['Transported']
features = dataset_df.drop('Transported', axis=1)

# Get categorical and numerical columns
categorical_cols = features.select_dtypes(include=['object']).columns.tolist()
numerical_cols = features.select_dtypes(include=['int64', 'float64']).columns.tolist()

print(f"Categorical columns: {categorical_cols}")
print(f"Numerical columns: {numerical_cols}")

In [ ]:
# One-hot encode categorical features
features_encoded = pd.get_dummies(features, columns=categorical_cols, drop_first=False)

# Fill any remaining NaN values with 0
features_encoded = features_encoded.fillna(0)

print(f"Features shape after encoding: {features_encoded.shape}")
features_encoded.head()

# Split Dataset into Train and Validation Sets

Following the 80-20 split from the original solution

In [ ]:
# Split data: 80% training, 20% validation
X_train, X_valid, y_train, y_valid = train_test_split(
    features_encoded, 
    label, 
    test_size=0.2, 
    random_state=42
)

print(f"Training set size: {X_train.shape[0]}")
print(f"Validation set size: {X_valid.shape[0]}")

# Train Random Forest Model

Using sklearn's RandomForestClassifier with similar parameters to TensorFlow Decision Forests

In [ ]:
# Initialize Random Forest Classifier
# n_estimators=300 matches the default number of trees in TFDF
rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1,  # Use all CPU cores
    oob_score=True,  # Enable Out-of-Bag scoring (similar to TFDF)
    verbose=1
)

print("Training Random Forest model...")
rf_model.fit(X_train, y_train)
print("Training complete!")

# Visualize a Single Tree

Similar to TFDF's tree visualization

In [ ]:
# Visualize the first tree with max_depth=3
plt.figure(figsize=(20, 10))
plot_tree(
    rf_model.estimators_[0], 
    max_depth=3, 
    feature_names=features_encoded.columns,
    class_names=['Not Transported', 'Transported'],
    filled=True,
    rounded=True
)
plt.title("Decision Tree (tree_idx=0, max_depth=3)")
plt.show()

# Evaluate Model on Out-of-Bag (OOB) and Validation Data

OOB score is similar to TFDF's OOB evaluation

In [ ]:
# Out-of-Bag Score
oob_score = rf_model.oob_score_
print(f"Out-of-Bag (OOB) Accuracy: {oob_score:.4f}")

In [ ]:
# Plot OOB accuracy vs number of trees
# Note: sklearn doesn't track this automatically, so we'll train incrementally
oob_scores = []
tree_counts = range(1, 301, 10)

for n_trees in tree_counts:
    rf_temp = RandomForestClassifier(
        n_estimators=n_trees,
        random_state=42,
        n_jobs=-1,
        oob_score=True
    )
    rf_temp.fit(X_train, y_train)
    oob_scores.append(rf_temp.oob_score_)

plt.figure(figsize=(10, 6))
plt.plot(tree_counts, oob_scores, marker='o')
plt.xlabel("Number of Trees")
plt.ylabel("Accuracy (Out-of-Bag)")
plt.title("OOB Accuracy vs Number of Trees")
plt.grid(True)
plt.show()

In [ ]:
# Validation Set Evaluation
y_valid_pred = rf_model.predict(X_valid)
valid_accuracy = accuracy_score(y_valid, y_valid_pred)

print(f"Validation Accuracy: {valid_accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(y_valid, y_valid_pred, target_names=['Not Transported', 'Transported']))

# Feature Importances

Similar to TFDF's variable importances

In [ ]:
# Get feature importances
feature_importances = pd.DataFrame({
    'feature': features_encoded.columns,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

print("Top 15 Most Important Features:")
print(feature_importances.head(15))

In [ ]:
# Visualize top 15 feature importances
plt.figure(figsize=(10, 8))
top_features = feature_importances.head(15)
plt.barh(top_features['feature'], top_features['importance'])
plt.xlabel('Importance Score')
plt.title('Top 15 Feature Importances')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

# Prepare Test Data and Make Predictions

In [ ]:
# Load test dataset
test_df = pd.read_csv('/kaggle/input/spaceship-titanic/test.csv')
submission_id = test_df.PassengerId.copy()

# Apply same preprocessing as training data
test_df = test_df.drop(['PassengerId', 'Name'], axis=1)

# Replace NaN values
test_df[['VIP', 'CryoSleep']] = test_df[['VIP', 'CryoSleep']].fillna(value=0)

# Extract Deck, Cabin_num, and Side
test_df[["Deck", "Cabin_num", "Side"]] = test_df["Cabin"].str.split("/", expand=True)
test_df = test_df.drop('Cabin', axis=1)

# Convert boolean to integers
test_df['VIP'] = test_df['VIP'].astype(int)
test_df['CryoSleep'] = test_df['CryoSleep'].astype(int)

# One-hot encode categorical features
test_encoded = pd.get_dummies(test_df, columns=categorical_cols, drop_first=False)

# Align test set with training set columns
test_encoded = test_encoded.reindex(columns=features_encoded.columns, fill_value=0)

print(f"Test set shape: {test_encoded.shape}")
test_encoded.head()

In [ ]:
# Make predictions
predictions_proba = rf_model.predict_proba(test_encoded)[:, 1]
predictions = (predictions_proba > 0.5).astype(bool)

# Create submission dataframe
output = pd.DataFrame({
    'PassengerId': submission_id,
    'Transported': predictions
})

output.head()

# Save Submission File

In [ ]:
# Save submission file
output.to_csv('/kaggle/working/submission.csv', index=False)
print("Submission file saved successfully!")
output.head(10)

# Summary

This notebook replicates the TensorFlow Decision Forests approach using PyTorch ecosystem and sklearn:

- **Data preprocessing**: Same feature engineering (Deck, Cabin_num, Side extraction)
- **Model**: RandomForestClassifier with 300 trees (matching TFDF defaults)
- **Evaluation**: OOB scoring and validation set evaluation
- **Feature importance**: Analyzed most important features
- **Predictions**: Generated submission file

Note: PyTorch doesn't have native decision forest support, so we used sklearn's implementation which is highly optimized and widely used in production.